# Exploratory Data Analysis

This notebook explores the MovieLens 100K dataset used in the recommendation system project.

The main goals are:

- understand the user, movie, and rating tables
- inspect interaction patterns
- check rating distribution
- explore temporal activity
- look at highly rated genre behavior

This analysis helps verify the data before training recommendation models.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data.download import download_movielens_100k
from src.data.preprocessing import load_movielens_data, clean_ratings, preprocess_tables

In [ ]:
dataset_dir = download_movielens_100k()
users_df, ratings_df, items_df, genres_df = load_movielens_data(dataset_dir)

ratings_df = clean_ratings(ratings_df, items_df)
users_processed, items_processed = preprocess_tables(users_df, items_df)

## Raw table shapes

In [ ]:
print("Users shape:", users_df.shape)
print("Ratings shape:", ratings_df.shape)
print("Items shape:", items_df.shape)

users_df.head()

In [ ]:
ratings_df.head()

In [ ]:
items_df.head()

## Missing values

In [ ]:
print("Users missing values:")
display(users_df.isna().sum())

print("\nRatings missing values:")
display(ratings_df.isna().sum())

print("\nItems missing values:")
display(items_df.isna().sum())

## Rating distribution

In [ ]:
rating_counts = ratings_df["rating"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(rating_counts.index, rating_counts.values, edgecolor="black")
plt.title("Distribution of Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.xticks(rating_counts.index)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.show()

rating_counts

## Ratings over time

In [ ]:
monthly_counts = ratings_df.groupby("rating_month")["rating"].count()

plt.figure(figsize=(10, 4))
monthly_counts.plot(kind="bar")
plt.title("Number of Ratings per Month")
plt.xlabel("Month")
plt.ylabel("Count")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.show()

monthly_counts.tail()

## Highly rated interactions
Here, highly rated means rating >= 4.

In [ ]:
high_ratings = ratings_df[ratings_df["rating"] >= 4].copy()
high_rating_counts = high_ratings.groupby("user_id")["movie_id"].count()

plt.figure(figsize=(7, 4))
plt.hist(high_rating_counts, bins=50, edgecolor="black")
plt.title("Distribution of High Ratings per User")
plt.xlabel("Count of ratings >= 4")
plt.ylabel("Number of users")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.show()

print("Average number of ratings >= 4 per user:", round(high_rating_counts.mean(), 2))

## Genre preference spread among users
This checks how many different genres appear in each user's highly rated movies.

In [ ]:
high_ratings_with_items = high_ratings.merge(items_processed, on="movie_id")

genre_cols = [
    c for c in high_ratings_with_items.columns
    if c not in [
        "user_id", "movie_id", "rating", "timestamp",
        "rating_timestamp", "rating_month", "release_date", "title"
    ]
]

user_genre_counts = high_ratings_with_items.groupby("user_id")[genre_cols].max().sum(axis=1)

plt.figure(figsize=(7, 4))
plt.hist(user_genre_counts, bins=20, edgecolor="black")
plt.title("Number of Highly Rated Genres per User")
plt.xlabel("Number of genres")
plt.ylabel("Number of users")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.show()

user_genre_counts.describe()

## Basic observations

Some quick observations from the dataset:

- ratings are concentrated around the middle-to-high values
- user activity is uneven
- some users rate many more items than others
- temporal behavior matters, so a random split would be weaker than a time-based split
- users often highly rate movies across multiple genres, which supports trying both collaborative and feature-based approaches